In [1]:
! pip install seaborn

In [2]:
 # Libs principais
from model import *
import pandas as pd
# Controle de execução e pastas
import os
# Desativar alguns warnings
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
plt.style.use('ggplot')

def VirtualLaneDetector(x,y,virtual_lane_lim,virtual_lane_width):
    virtual_lane = -1
    count = 0

    for list_point in virtual_lane_lim:
        count = count + 1
        func_virtual_lane = PolygonalToFunction(list_point)
        if (y>=func_virtual_lane(x)-0.5*virtual_lane_width) and (y<=func_virtual_lane(x)+0.5*virtual_lane_width):
            virtual_lane = count
            break

    return virtual_lane

In [3]:
os.chdir("data_ignore")

In [4]:
df_via = pd.read_excel("data/Dados dos vídeos consolidados.xlsx",sheet_name="Coletas") 
df_video = pd.read_excel("data/Dados dos vídeos consolidados.xlsx",sheet_name="Vídeos")

In [5]:
df_run = df_video.merge(df_via,on="id_coleta",how="left")
df_run["cod_faixas"] = df_run["cod_faixas"].astype(str)

In [6]:
df_run

,id_video,id_coleta,processado_x,tratado_x,duracao_x,img_ref,mpp,limite_faixa,observacao_x,verificado,...,faixa_azul,largura_secao,largura_secao_px,larg_motofaixa,larg_motofaixa_px,fluxo,num_pista,cod_faixas,cod_motofaixa,observacao_y
0,Piloto1_Drone1_0004,40_Pil_SMF_MQ,True,False,NaN,Piloto1_Drone1_0004-419.png,NaN,"[[[884, 187], [1, 245]], [[698, 344], [0, 381]...",Vídeo vertical,1.0,...,Não,15.314752,NaN,0.0,NaN,←,1.0,"1,2,3,4,5",3,Vídeo exigiu correções de ângulação mínimas pr...
1,Piloto1_Drone1_0005,40_Pil_SMF_MQ,True,False,NaN,Piloto1_Drone1_0005-65.png,NaN,"[[[846, 193], [1, 264]], [[668, 348], [0, 396]...",Vídeo vertical,1.0,...,Não,15.314752,NaN,0.0,NaN,←,1.0,"1,2,3,4,5",3,Vídeo exigiu correções de ângulação mínimas pr...
2,Piloto1_Drone1_0007,40_Pil_SMF_MQ,True,False,NaN,Piloto1_Drone1_0007-225.png,NaN,"[[[894, 133], [1, 135]], [[700, 276], [0, 269]...",Vídeo vertical,1.0,...,Não,15.314752,NaN,0.0,NaN,←,1.0,"1,2,3,4,5",3,Vídeo exigiu correções de ângulação mínimas pr...
3,Piloto1_Drone1_0008,40_Pil_SMF_MQ,True,False,NaN,Piloto1_Drone1_0008-45.png,NaN,"[[[846, 163], [1, 201]], [[658, 315], [0, 337]...",Vídeo vertical,1.0,...,Não,15.314752,NaN,0.0,NaN,←,1.0,"1,2,3,4,5",3,Vídeo exigiu correções de ângulação mínimas pr...
4,Piloto1_Drone1_0009,40_Pil_SMF_MQ,True,True,0.230833,Piloto1_Drone1_0009-225.png,0.028393,"[[[1627, 262], [1892, 249]], [[32, 468], [1794...",NaN,1.0,...,Não,15.314752,NaN,0.0,NaN,←,1.0,"1,2,3,4,5",3,Vídeo exigiu correções de ângulação mínimas pr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,81_B_1,81,True,True,0.230851,81_B_1-424.png,0.031356,"[[80, 199], [199, 286], [286, 375], [375, 472]...",NaN,1.0,...,Não,11.470382,NaN,0.0,NaN,←,2.0,"1,2,3,4",4,NaN
222,81_B_2,81,True,True,0.052877,81_B_1-424.png,0.031356,"[[80, 199], [199, 286], [286, 375], [375, 472]...",NaN,1.0,...,Não,11.470382,NaN,0.0,NaN,←,2.0,"1,2,3,4",4,NaN
223,81_B_3,81,True,True,0.133458,81_B_3-251.png,0.029603,"[[84, 208], [208, 301], [301, 396], [396, 496]...",NaN,1.0,...,Não,11.470382,NaN,0.0,NaN,←,2.0,"1,2,3,4",4,NaN
224,81_B_4,81,True,True,0.230851,81_B_4-565.png,0.030825,"[[88, 203], [203, 292], [292, 383], [383, 478]...",NaN,1.0,...,Não,11.470382,NaN,0.0,NaN,←,2.0,"1,2,3,4",4,NaN


In [7]:
for index,row in df_run.iterrows():
    try:
        print(f"Processando {row['id_video']}")
        model = YoloMicroscopicDataProcessing()
        model.ImportFromJSON(f"data/json/{row['id_video']}.json")

        # Agregar volumes
        # Filtrar faixas
        valid_lanes = row["cod_faixas"].split(",")
        model.df = model.df[model.df["traffic_lane"].isin(valid_lanes)]
        print(model.df)

        # Agregar volumes
        df_agg = model.df.groupby("id").agg({
            "traffic_lane":pd.Series.mode,
            "vehicle_type":pd.Series.mode})
        
        df_agg = df_agg.reset_index(drop=True)
        df_agg["traffic_lane"] = df_agg["traffic_lane"].astype("str")
        df_agg["vehicle_type"] = df_agg["vehicle_type"].astype("str")
        df_agg["vehicle_count"] = 1
        df_agg = df_agg.groupby(["traffic_lane","vehicle_type"]).agg({"vehicle_count":"sum"})
        df_agg = df_agg.reset_index(drop=False)
        df_agg.insert(0,"file",row['id_video'])
        df_agg["time"] = model.df["instant"].max()/3600

        df_agg.to_csv(f"data/collected/count_flow/{row['id_video']}.csv",index=False)

        print(f"OK {row['id_video']}")

    except Exception as e:
        print(row["id_video"],e)

Processando Piloto1_Drone1_0004
Piloto1_Drone1_0004 [Errno 2] No such file or directory: 'data/json/Piloto1_Drone1_0004.json'
Processando Piloto1_Drone1_0005
Piloto1_Drone1_0005 [Errno 2] No such file or directory: 'data/json/Piloto1_Drone1_0005.json'
Processando Piloto1_Drone1_0007
Piloto1_Drone1_0007 [Errno 2] No such file or directory: 'data/json/Piloto1_Drone1_0007.json'
Processando Piloto1_Drone1_0008
Piloto1_Drone1_0008 [Errno 2] No such file or directory: 'data/json/Piloto1_Drone1_0008.json'
Processando Piloto1_Drone1_0009
Empty DataFrame
Columns: [frame, p1xbb, p1ybb, p2xbb, p2ybb, id, conf_YOLO, vehicle_type, traffic_lane, instant, vehicle_length, vehicle_width, y, x, tail, head, global_id, x_instant_speed, y_instant_speed, instant_speed, x_instant_acc, y_instant_acc, instant_acc]
Index: []

[0 rows x 23 columns]
OK Piloto1_Drone1_0009
Processando Piloto1_Drone1_0010
Empty DataFrame
Columns: [frame, p1xbb, p1ybb, p2xbb, p2ybb, id, conf_YOLO, vehicle_type, traffic_lane, instant

In [11]:
vva = [4]
traffic_lanes_valid = [1,2,3,4]
id = "81"

root_file = "data/json"
all_files = os.listdir(root_file)
all_files = [i for i in all_files if id in i]

for f in all_files:
    # try:
    print(f"Processando {f}")
    model = YoloMicroscopicDataProcessing()
    model.ImportFromJSON(f"data/json/{f}")

    # Mantém só uma das pistas
    model.df = model.df[model.df[model.traffic_lane_column].isin(traffic_lanes_valid)]

    # Remover dados das extreminades do vídeo
    model.df = model.df[model.df['x'].between(3,model.video_width-3)]

    # Alterar a frequência da amostra para 10fps
    min_frame = model.df[model.frame_column].min()
    max_frame = model.df[model.frame_column].max()
    list_frames = list(range(min_frame,max_frame+1,3))
    model.df = model.df[model.df[model.frame_column].isin(list_frames)]

    # Dataframe de motocicletas
    df_motorcycle = model.df[model.df[model.vehicle_type_column].isin(['Moto'])].sort_values([model.frame_column,model.id_column])
    df_motorcycle.insert(0,'id_voo',f.split(".")[0])

    # Definição das faixas de tráfego
    # Centroide (y) dentro da região cujo centroo é o limite entre as faixas
    # Largura teórica do corredor (metros), corresponde a largura de 2 motos
    virtual_lane_width = 1.6
    # Corredor sem faixa azul
    virutal_lane_group = {'Corredor Principal':vva}
    virutal_lane_group["Outros Corredores"] = [i for i in  range(1,len(model.virtual_lane_lim)+1) if i not in virutal_lane_group['Corredor Principal']]

    # Qual corredor virual pertence (-1 para nenhum corredor)
    df_motorcycle['virutal_lane'] = df_motorcycle.apply(lambda row:VirtualLaneDetector(row[model.x_centroid_column],row[model.y_centroid_column],model.virtual_lane_lim,virtual_lane_width),axis=1)
    # Corredores não classificados recebem 0
    df_motorcycle['zero_temp'] = ((-df_motorcycle['virutal_lane'].isin(JoinList(list(virutal_lane_group.values())))) & (df_motorcycle['virutal_lane']!=-1))
    df_motorcycle['virutal_lane'] = df_motorcycle.apply(lambda x:x['virutal_lane'] if not x['zero_temp'] else 0,axis=1)
    df_motorcycle = df_motorcycle.drop(columns=['zero_temp'])

    # Tipo/nome do corredor
    df_motorcycle['virtual_lane_type'] = np.nan
    for key,value in virutal_lane_group.items():
        df_motorcycle['virtual_lane_type'] = df_motorcycle.apply(lambda x:key if x['virutal_lane'] in value else x['virtual_lane_type'],axis=1)
    # Se for -1, estava na mais centralizado na faixa de tráfefo misto
    df_motorcycle['virtual_lane_type'] = df_motorcycle.apply(lambda x:'Fora do Corredor' if x['virutal_lane']==-1 else x['virtual_lane_type'],axis=1)
    # Se estava em outras faixas não avaliadas
    df_motorcycle['virtual_lane_type'] = df_motorcycle.apply(lambda x:'Outro Corredor' if x['virutal_lane']==0 else x['virtual_lane_type'],axis=1)

    # Condição do tráfego
    speed_ref = '85%'
    agg_traffic_state = model.df[-model.df[model.vehicle_type_column].isin(['Moto'])].groupby(['frame'])['instant_speed'].describe(percentiles=[0.85])
    agg_traffic_state = agg_traffic_state.reset_index(drop=False)
    df_motorcycle = df_motorcycle.merge(agg_traffic_state[['frame',speed_ref]],on='frame',how='left')
    df_motorcycle['traffic_condition_speed'] = df_motorcycle[speed_ref]*3.6
    df_motorcycle = df_motorcycle.drop(columns=[speed_ref])
    df_motorcycle['traffic_condition'] = df_motorcycle['traffic_condition_speed'].apply(lambda x:'Congestionado' if x<5 else 'Não Congestionado')

    # Distância lateral
    side_distance = pd.DataFrame()
    for motorcycle_id in df_motorcycle[model.id_column].unique().tolist():
        frame_list = df_motorcycle[df_motorcycle[model.id_column]==motorcycle_id].sort_values(model.frame_column)[model.frame_column].tolist()
        df_side = pd.concat([model.SideVehicle(
            motorcycle_id,
            t,
            overlap_lon=0.3,
            overlap_lat=0.3,
            report_just_min=True,
            ) for t in frame_list],ignore_index=True)
        df_side = df_side.rename(columns=dict(zip(df_side.columns, [i+'_vehicle_side' if 'speed' in i else i for i in df_side.columns])))
        df_side.insert(0,'id_motorcycle',motorcycle_id)
        side_distance = pd.concat([side_distance,df_side],ignore_index=True)

    side_distance = side_distance.rename(columns={'id':'id_vehicle_side'})
    side_distance[model.global_id_column] = side_distance['id_motorcycle'].astype(str) + '@' + side_distance[model.frame_column].astype(str)
    df_motorcycle = df_motorcycle.merge(side_distance[[
        'global_id',
        'id_vehicle_side',
        'x_instant_speed_vehicle_side',
        'y_instant_speed_vehicle_side',
        'instant_speed_vehicle_side',
        'lateral_distance_between_vehicles',
        'side'
        ]],on=model.global_id_column,how='left')

    df_motorcycle['delta_speed_vehicle_side'] = df_motorcycle[model.instant_speed_column] - df_motorcycle[model.instant_speed_column+'_vehicle_side']
    df_motorcycle['delta_speed_vehicle_side'] = df_motorcycle['delta_speed_vehicle_side']

    df_headway = []
    for index,row in df_motorcycle.iterrows():
        try:
            hd = model.HeadwayDeltaSpeed(row["id"],row["frame"])
            if not hd.empty:
                df_headway.append(hd)
        except Exception as e:
            print("Deu ruim pulou",row["id"],row["frame"])
    print("OK")
    
    if len(df_headway)>0:
        df_headway = pd.concat(df_headway,ignore_index=True)
        df_headway["id"] = df_headway["id_follower"]
        df_motorcycle = df_motorcycle.merge(df_headway,on=["frame","id"],how="left")

    df_motorcycle.to_csv(f"data/DistLatVelAceHeadway/{f.split('.')[0]}.csv",index=False)
    print(f"Fim {f}")
    # except Exception as e:
    #     print(f"Erro {f}")
    #     print(e)
        

Processando 81_A_1.json
Deu ruim pulou 954 3861
Deu ruim pulou 947 3915
Deu ruim pulou 840 4200
Deu ruim pulou 3351 16701
OK
Fim 81_A_1.json
Processando 81_A_2.json
Deu ruim pulou 1079 5826
Deu ruim pulou 1348 7785
OK
Fim 81_A_2.json
Processando 81_A_3.json
OK
Fim 81_A_3.json
Processando 81_A_4.json
OK
Fim 81_A_4.json
Processando 81_A_5.json
Deu ruim pulou 61 306
Deu ruim pulou 183 840
Deu ruim pulou 785 4851
Deu ruim pulou 875 5436
Deu ruim pulou 939 7203
Deu ruim pulou 1094 7203
Deu ruim pulou 1122 7203
OK
Fim 81_A_5.json
Processando 81_B_1.json
Deu ruim pulou 909 2082
Deu ruim pulou 1104 2319
Deu ruim pulou 2105 4419
Deu ruim pulou 2082 4683
Deu ruim pulou 2014 5853
Deu ruim pulou 2470 6399
Deu ruim pulou 2868 8727
Deu ruim pulou 2875 8832
Deu ruim pulou 3702 13536
Deu ruim pulou 4211 16071
Deu ruim pulou 4709 19023
Deu ruim pulou 5361 22218
OK
Fim 81_B_1.json
Processando 81_B_2.json
Deu ruim pulou 298 1329
Deu ruim pulou 300 1341
Deu ruim pulou 406 1809
Deu ruim pulou 575 3084
Deu 

In [16]:
root_path = "data/DistLatVelAceHeadway"
list_files = os.listdir(root_path)
df = []
for f in list_files:
    df_ = pd.read_csv(os.path.join(root_path,f))
    df.append(df_)

df = pd.concat(df,ignore_index=True)


In [18]:
# df = pd.read_csv(f"data/DistLatVelAceHeadway/{f.split('.')[0]}.csv")

# Dados filtrados
df_hist_speed_by_id = df.copy()
df_hist_speed_by_id['instant_speed'] = df_hist_speed_by_id['instant_speed']*3.6
df_hist_speed_by_id = df_hist_speed_by_id.groupby(['id_voo','traffic_condition','virtual_lane_type','id'])['instant_speed'].describe(percentiles=[0.05,0.25,0.5,0.75,0.85,0.95]).reset_index(drop=False)

# Pelo menos 1,5s na faixa de referencia
mask1 = df_hist_speed_by_id['count']>=15
df_hist_speed_by_id = df_hist_speed_by_id[mask1]
df_hist_speed_by_id

,id_voo,traffic_condition,virtual_lane_type,id,count,mean,std,min,5%,25%,50%,75%,85%,95%,max
0,10_A_1,Não Congestionado,Corredor Principal,62,27.0,75.014606,0.980702,73.813882,73.916772,74.094515,74.984874,75.622020,75.939237,76.800378,77.041716
1,10_A_1,Não Congestionado,Corredor Principal,79,19.0,83.933875,0.703032,82.636129,82.765290,83.486109,84.085216,84.470010,84.647990,84.834606,84.912811
3,10_A_1,Não Congestionado,Corredor Principal,220,15.0,77.898212,0.560033,77.082781,77.131431,77.337431,77.973920,78.472797,78.499543,78.509711,78.515558
5,10_A_1,Não Congestionado,Corredor Principal,302,21.0,89.842108,3.601871,83.211916,83.846263,87.109624,91.161726,92.386896,93.349399,94.136703,94.528365
7,10_A_1,Não Congestionado,Corredor Principal,314,31.0,76.137072,3.706002,70.318045,70.530452,72.679687,77.196579,79.102579,79.488548,80.900385,81.522305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8773,Piloto2_Drone2_0007,Não Congestionado,Corredor Principal,196,15.0,59.675507,0.943448,58.139499,58.207226,58.844964,60.179430,60.466335,60.514571,60.570249,60.619998
8779,Piloto2_Drone2_0007,Não Congestionado,Fora do Corredor,97,15.0,38.020722,1.034208,36.379468,36.499147,37.193702,38.087063,38.859439,39.165948,39.333828,39.526709
8781,Piloto2_Drone2_0007,Não Congestionado,Fora do Corredor,108,17.0,29.951325,2.049076,27.522151,27.719440,28.077007,29.601782,31.401132,32.318145,33.303344,33.775423
8782,Piloto2_Drone2_0007,Não Congestionado,Fora do Corredor,115,15.0,33.529366,1.078303,32.198427,32.266663,32.689450,33.233873,34.323732,34.836530,35.276041,35.419468


In [19]:
df_hist_speed_by_id.to_excel("Dados_23_07_25.xlsx",index=False)

#### Suavizar

In [5]:
if __name__=="__main__":
    # os.chdir("data_ignore")
    root_file = "data/json"
    all_files = os.listdir(root_file)

    for f in all_files:
        try:
            print(f"Processando {f}")
            model = YoloMicroscopicDataProcessing()
            model.ImportFromJSON(os.path.join(root_file,f))
        
            model_smoothed = model.SmoothingSavGolFilter(window_length=15,polyorder=1) 
            model_smoothed.to_csv(f"data/processed_smoothed/{f.replace('json','csv')}",index=False)
            print(f"Fim {f}")
        except Exception as e:
            print(f"Erro {f}")
            print(e)


Processando 10_A_1.json
Fim 10_A_1.json
Processando 10_A_2.json
Fim 10_A_2.json
Processando 10_A_3.json
Fim 10_A_3.json
Processando 10_A_4.json
Fim 10_A_4.json
Processando 10_A_5.json
Fim 10_A_5.json
Processando 10_B_1.json
Fim 10_B_1.json
Processando 10_B_2.json
Fim 10_B_2.json
Processando 10_B_3.json
Fim 10_B_3.json
Processando 10_B_4.json
Fim 10_B_4.json
Processando 10_B_5.json
Fim 10_B_5.json
Processando 32_A_1.json
Fim 32_A_1.json
Processando 32_A_2.json
Fim 32_A_2.json
Processando 32_A_3.json
Fim 32_A_3.json
Processando 32_A_4.json
Fim 32_A_4.json
Processando 32_A_5.json
Fim 32_A_5.json
Processando 32_B_1.json
Fim 32_B_1.json
Processando 32_B_2.json
Fim 32_B_2.json
Processando 32_B_3.json
Fim 32_B_3.json
Processando 32_B_4.json
Fim 32_B_4.json
Processando 32_B_5.json
Fim 32_B_5.json
Processando 79_A_1.json
Fim 79_A_1.json
Processando 79_A_2.json
Fim 79_A_2.json
Processando 79_A_3.json
Fim 79_A_3.json
Processando 79_A_4.json
Fim 79_A_4.json
Processando 79_A_5.json
Fim 79_A_5.json
